# 03 · Strategia: consigliata vs reale
Stima il degrado, calcola la strategia ottimale e confrontala con quella davvero usata in gara.

In [ ]:
import sys; sys.path.insert(0, '../src')
from f1va import data as f1data, features, models, strategy
import pandas as pd, numpy as np

In [ ]:
ses = f1data.load_session(2024, 'Monza', 'R')
laps = f1data.quicklaps(f1data.laps_dataframe(ses))
deg = features.degradation_table(laps)
tyre = strategy.fit_tyre_models(deg)
deg

## Strategia ottimale secondo il modello

In [ ]:
TOTAL_LAPS = 53
best = strategy.optimize_strategy(TOTAL_LAPS, list(tyre.keys()), tyre, max_stops=2)
print('Piano:', ' -> '.join(f'{c}({n})' for c,n in best.stints))
print('Tempo gara stimato:', round(best.total_time_s/60,2), 'min')

## Strategia reale del vincitore
Estrae gli stint realmente adottati dal pilota in testa e confronta.

In [ ]:
full = f1data.laps_dataframe(ses)
winner = full.Driver.iloc[full.groupby('Driver').LapNumber.max().argmax()]
real = (full[full.Driver==winner].groupby('Stint')
        .agg(compound=('Compound','first'), laps=('LapNumber','count')))
print('Pilota:', winner)
real

## Undercut
Quanto conviene anticipare la sosta su gomma nuova.

In [ ]:
keys=list(tyre.keys())
print('Undercut gain (s):', strategy.undercut_gain(tyre, keys[0], rival_deg_compound=keys[-1]))